<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 PAI-Bench-C (Conditional Generation) Reproduction with Cosmos Framework

This notebook walks through generating the [PAI-Bench Conditional Generation (PAI-Bench-C)](https://huggingface.co/datasets/shi-labs/physical-ai-bench-conditional-generation) set with Cosmos3-Super using the native Cosmos Framework PyTorch entrypoint:

```bash
python -m cosmos_framework.scripts.inference
```

PAI-Bench-C evaluates video-to-video transfer across **600 tasks** and **four control modalities**:

| Modality | Control input | Primary metric | Direction |
| --- | --- | --- | --- |
| **edge** | Canny edge map (`canny/task_XXXX.mp4`) | Canny F1 | ↑ higher is better |
| **blur** | Blurred reference (`blur/task_XXXX.mp4`) | Blur SSIM | ↑ higher is better |
| **depth** | Depth map (`depth_vids/task_XXXX.mp4`) | Depth SI-RMSE | ↓ lower is better |
| **seg** | Segmentation map (`sam2_vids/task_XXXX.mp4`) | Seg mIoU | ↑ higher is better |

All modalities generate 121 frames at 30 FPS, 720p. After generation, metrics are computed with the open-source [`physical-ai-bench`](https://github.com/SHI-Labs/physical-ai-bench) library.

**Sections:**
1. Configure paths and environment
2. Clone or reuse Cosmos Framework
3. Install native PyTorch dependencies
4. Verify GPU and Python environment
5. Download the PAI-Bench-C dataset
6. Load prompts and preview the demo case
7. Helper functions
8. Demo case — run one task with one modality
9. (Optional) Full sweep — edge (600 tasks)
10. (Optional) Full sweep — blur (600 tasks)
11. (Optional) Full sweep — depth (600 tasks)
12. (Optional) Full sweep — seg (600 tasks)
13. Evaluate generated videos

## 1. Configure Paths and Environment

All paths default to sensible locations under this `cosmos` checkout. Override any of them by exporting before launching the notebook:

```bash
export HF_TOKEN=hf_...               # required: HuggingFace token for model + dataset access
export COSMOS3_REPO=/path/to/cosmos-framework
export COSMOS3_UV_GROUP=cu130-train   # or cu128-train
export UV_PROJECT_ENVIRONMENT=/path/to/large/uv/venvs/cosmos3-paibench-c
export UV_CACHE_DIR=/path/to/large/uv/cache       # avoids home-dir disk quota issues
export HF_HOME=/path/to/large/huggingface/cache   # avoids home-dir disk quota issues
export TRITON_CACHE_DIR=/path/to/large/triton/cache  # avoids home-dir disk quota issues (~/.triton)
export COSMOS3_NUM_GPUS=4
export CUDA_VISIBLE_DEVICES=0,1,2,3
export PAIBENCH_C_DATASET_ROOT=/path/to/physical-ai-bench-conditional-generation
export PAIBENCH_C_OUTPUT_ROOT=/path/to/paibench_c/outputs
export PAIBENCH_C_CHECKPOINT=Cosmos3-Super   # default: Cosmos3-Nano
```

**HuggingFace auth**: set `HF_TOKEN` (recommended), or run `uvx hf auth login` once — the token is then cached in `HF_HOME`. `UV_CACHE_DIR`, `HF_HOME`, and `TRITON_CACHE_DIR` all default to `.agents/skills/evaluation/cosmos3/generator/paibench_c/.cache/{uv,huggingface,triton}` (lustre) to avoid home-dir quota issues. Set `COSMOS3_CACHE_ROOT` to redirect all three at once.

In [ ]:
from pathlib import Path
import json
import os
import shutil


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "cookbooks" / "cosmos3").is_dir():
            return path
    return start


def default_framework_repo(root: Path) -> Path:
    for candidate in (root / "packages" / "cosmos-framework", root / "packages" / "cosmos3"):
        if (candidate / "pyproject.toml").exists() and (candidate / "cosmos_framework").exists():
            return candidate
    return root / "packages" / "cosmos-framework"


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", default_framework_repo(COSMOS_ROOT))).resolve()
COSMOS3_GIT_URL = os.environ.get("COSMOS3_GIT_URL", "https://github.com/NVIDIA/cosmos-framework.git")
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    print("WARNING: HF_TOKEN is not set. Model and dataset downloads will fail for gated repos.")
    print("  Option 1 (recommended): export HF_TOKEN=hf_... before launching the notebook.")
    print("  Option 2: run `uvx hf auth login` once (stores token in HF_HOME).")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print(f"HF_TOKEN: {'*' * (len(HF_TOKEN) - 4)}{HF_TOKEN[-4:]}")

COSMOS3_UV_GROUP = os.environ.get("COSMOS3_UV_GROUP", "cu130-train")
COSMOS3_UV_ENV = Path(os.environ.get("UV_PROJECT_ENVIRONMENT", COSMOS3_REPO / ".venv")).resolve()
PAIBENCH_C_NOTEBOOK_ROOT = COSMOS_ROOT / "evaluation" / "cosmos3" / "generator" / "paibench_c"
PAIBENCH_C_ROOT = PAIBENCH_C_NOTEBOOK_ROOT  # alias used in evaluation cells
_cache_root = Path(
    os.environ.get("COSMOS3_CACHE_ROOT", PAIBENCH_C_NOTEBOOK_ROOT / ".cache")
).resolve()


def _default_cache_path(name: str) -> Path:
    """Resolve cache directory under COSMOS3_CACHE_ROOT or notebook .cache/."""
    if "COSMOS3_CACHE_ROOT" in os.environ:
        return (Path(os.environ["COSMOS3_CACHE_ROOT"]).expanduser() / name).resolve()
    return (_cache_root / name).resolve()


# Redirect all caches to lustre (avoids home-dir disk-quota issues on ~/.triton, ~/.cache/uv, etc.).
# Override via COSMOS3_CACHE_ROOT or the individual env vars.
UV_CACHE_DIR = Path(os.environ.get("UV_CACHE_DIR", _default_cache_path("uv")))
HF_HOME = Path(os.environ.get("HF_HOME", _default_cache_path("huggingface")))
TRITON_CACHE_DIR = Path(os.environ.get("TRITON_CACHE_DIR", _default_cache_path("triton")))
COSMOS3_NUM_GPUS = os.environ.get("COSMOS3_NUM_GPUS", "4")
CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1,2,3")
PAIBENCH_C_ASSETS = PAIBENCH_C_NOTEBOOK_ROOT / "assets"
PROMPTS_FILE = PAIBENCH_C_ASSETS / "prompts.json"

PAIBENCH_C_HF_URL = os.environ.get(
    "PAIBENCH_C_HF_URL",
    "https://huggingface.co/datasets/shi-labs/physical-ai-bench-conditional-generation",
)
PAIBENCH_C_DATASET_ROOT = Path(
    os.environ.get(
        "PAIBENCH_C_DATASET_ROOT",
        PAIBENCH_C_NOTEBOOK_ROOT / "physical-ai-bench-conditional-generation",
    )
).resolve()
PAIBENCH_C_OUTPUT_ROOT = Path(
    os.environ.get("PAIBENCH_C_OUTPUT_ROOT", PAIBENCH_C_NOTEBOOK_ROOT / "outputs")
).resolve()
PAIBENCH_C_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Demo: run N tasks with a single modality (1 = quick smoke-test, 4 = multi-GPU test)
DEMO_TASK_ID = os.environ.get("PAIBENCH_C_DEMO_TASK", "task_0000")
DEMO_NUM_TASKS = int(os.environ.get("PAIBENCH_C_DEMO_NUM_TASKS", "1"))
DEMO_MODALITY = os.environ.get("PAIBENCH_C_DEMO_MODALITY", "edge")
CHECKPOINT = os.environ.get("PAIBENCH_C_CHECKPOINT", "Cosmos3-Nano")

for key, value in [
    ("COSMOS_ROOT", COSMOS_ROOT),
    ("COSMOS3_REPO", COSMOS3_REPO),
    ("COSMOS3_UV_ENV", COSMOS3_UV_ENV),
    ("UV_CACHE_DIR", UV_CACHE_DIR),
    ("HF_HOME", HF_HOME),
    ("TRITON_CACHE_DIR", TRITON_CACHE_DIR),
    ("COSMOS3_NUM_GPUS", COSMOS3_NUM_GPUS),
    ("CUDA_VISIBLE_DEVICES", CUDA_VISIBLE_DEVICES),
    ("PAIBENCH_C_DATASET_ROOT", PAIBENCH_C_DATASET_ROOT),
    ("PAIBENCH_C_OUTPUT_ROOT", PAIBENCH_C_OUTPUT_ROOT),
    ("DEMO_TASK_ID", DEMO_TASK_ID),
    ("DEMO_NUM_TASKS", DEMO_NUM_TASKS),
    ("DEMO_MODALITY", DEMO_MODALITY),
    ("CHECKPOINT", CHECKPOINT),
]:
    print(f"{key}={value}")

os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_GIT_URL"] = COSMOS3_GIT_URL
os.environ["COSMOS3_UV_GROUP"] = COSMOS3_UV_GROUP
os.environ["COSMOS3_UV_ENV"] = str(COSMOS3_UV_ENV)
os.environ["UV_PROJECT_ENVIRONMENT"] = str(COSMOS3_UV_ENV)
os.environ["UV_CACHE_DIR"] = str(UV_CACHE_DIR)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TRITON_CACHE_DIR"] = str(TRITON_CACHE_DIR)
os.environ["COSMOS3_NUM_GPUS"] = COSMOS3_NUM_GPUS
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["PAIBENCH_C_HF_URL"] = PAIBENCH_C_HF_URL
os.environ["PAIBENCH_C_DATASET_ROOT"] = str(PAIBENCH_C_DATASET_ROOT)
os.environ["PAIBENCH_C_OUTPUT_ROOT"] = str(PAIBENCH_C_OUTPUT_ROOT)
os.environ["DEMO_TASK_ID"] = DEMO_TASK_ID
os.environ["DEMO_NUM_TASKS"] = str(DEMO_NUM_TASKS)
os.environ["DEMO_MODALITY"] = DEMO_MODALITY
os.environ["CHECKPOINT"] = CHECKPOINT
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["COSMOS3_CACHE_ROOT"] = str(_cache_root)
os.environ["PAIBENCH_C_ROOT"] = str(PAIBENCH_C_ROOT)

# -- Auto-install uv if not on PATH -------------------------------------------
import shutil as _shutil, subprocess as _subprocess

if not _shutil.which("uv"):
    _uv_bin_dir = _cache_root / "uv-bin"
    _uv_bin_dir.mkdir(parents=True, exist_ok=True)
    print(f"uv not found - installing to {_uv_bin_dir} ...")
    _subprocess.run(
        ["sh", "-c",
         f'UV_INSTALL_DIR="{_uv_bin_dir}" curl -LsSf https://astral.sh/uv/0.11.29/install.sh | sh'],
        check=True,
    )
    os.environ["PATH"] = str(_uv_bin_dir / "bin") + ":" + os.environ.get("PATH", "")
    if not _shutil.which("uv"):
        raise RuntimeError("uv install failed - check network access and retry")
print(f"uv: {_shutil.which('uv')}")

# -- Auto-install git-lfs if not on PATH --------------------------------------
# git-lfs must be on PATH before any uv sync / git clone cells, because uv
# clones git-sourced dependencies (e.g. lerobot) whose repos have LFS hooks.
# Without the binary, git's filter-process fails even with GIT_LFS_SKIP_SMUDGE=1.
if not _shutil.which("git-lfs"):
    import platform as _platform
    _lfs_bin_dir = _cache_root / "bin"
    _lfs_bin_dir.mkdir(parents=True, exist_ok=True)
    _lfs_arch = "amd64" if _platform.machine() == "x86_64" else "arm64"
    _lfs_ver = "3.6.1"
    _lfs_url = (
        f"https://github.com/git-lfs/git-lfs/releases/download"
        f"/v{_lfs_ver}/git-lfs-linux-{_lfs_arch}-v{_lfs_ver}.tar.gz"
    )
    print(f"git-lfs not found - downloading v{_lfs_ver} to {_lfs_bin_dir} ...")
    _subprocess.run(
        ["sh", "-c",
         f'curl -LsSf "{_lfs_url}"'
         f' | tar -xz -C "{_lfs_bin_dir}" --strip-components=1'
         f' "git-lfs-{_lfs_ver}/git-lfs" 2>/dev/null'
         f' || curl -LsSf "{_lfs_url}"'
         f' | tar -xz -C "{_lfs_bin_dir}" --wildcards --strip-components=1 "*/git-lfs"'],
        check=True,
    )
    (_lfs_bin_dir / "git-lfs").chmod(0o755)
    os.environ["PATH"] = str(_lfs_bin_dir) + ":" + os.environ.get("PATH", "")
    if not _shutil.which("git-lfs"):
        raise RuntimeError("git-lfs install failed - install manually: https://git-lfs.com/")
print(f"git-lfs: {_shutil.which('git-lfs')}")

# -- Git hardening: non-interactive + trust all dirs --------------------------
# GIT_TERMINAL_PROMPT=0 prevents git from blocking on credential prompts in
# headless/nbconvert runs.  The GIT_CONFIG_COUNT vars tell git to treat every
# directory as safe, avoiding "dubious ownership" errors when running as root
# on Lustre paths owned by another user (git >= 2.35.2).
os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")
os.environ.setdefault("GIT_CONFIG_COUNT", "1")
os.environ.setdefault("GIT_CONFIG_KEY_0", "safe.directory")
os.environ.setdefault("GIT_CONFIG_VALUE_0", "*")

## 2. Clone or Reuse Cosmos Framework

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -f "$COSMOS3_REPO/pyproject.toml" ] && [ -d "$COSMOS3_REPO/cosmos_framework" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
elif [ -e "$COSMOS3_REPO" ]; then
  echo "COSMOS3_REPO exists but is not a Cosmos Framework checkout: $COSMOS3_REPO"
  exit 1
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git -c credential.helper= clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

echo "Framework checkout: $COSMOS3_REPO"

## 3. Install Native PyTorch Dependencies

Installs framework dependencies with the requested CUDA group (default `cu130-train`).

In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo "uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/"
  exit 1
fi

# A venv is considered healthy only if the python binary exists AND key packages
# (pydantic, torch) are importable.  An orphaned or partially-installed venv that
# passes the binary check but fails imports will trigger a fresh uv sync.
_venv_healthy() {
  [[ -x "$COSMOS3_UV_ENV/bin/python" ]] || return 1
  "$COSMOS3_UV_ENV/bin/python" -c 'import pydantic, torch' 2>/dev/null
}

if _venv_healthy && [[ "${FORCE_INSTALL:-0}" != "1" ]]; then
  echo "Existing venv found at $COSMOS3_UV_ENV — skipping uv sync (set FORCE_INSTALL=1 to reinstall)"
else
  export GIT_LFS_SKIP_SMUDGE=1
  cd "$COSMOS3_REPO"
  export UV_PROJECT_ENVIRONMENT="${UV_PROJECT_ENVIRONMENT:-$COSMOS3_UV_ENV}"
  echo "Running uv sync (group=$COSMOS3_UV_GROUP) ..."
  uv sync --all-extras --group="$COSMOS3_UV_GROUP"
  if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
    echo "uv sync completed, but expected Python is missing: $COSMOS3_UV_ENV/bin/python"
    exit 1
  fi
fi

# pynvml is listed in pyproject.toml with sys_platform=='never' (intentionally excluded from
# uv sync), but cosmos_framework/utils/distributed.py imports it unconditionally at module
# level.  Install it explicitly so inference workers don't fail with ModuleNotFoundError.
if ! "$COSMOS3_UV_ENV/bin/python" -c 'import pynvml' 2>/dev/null; then
  echo "Installing pynvml into cosmos3 venv ..."
  UV_PROJECT_ENVIRONMENT="$COSMOS3_UV_ENV" uv pip install --python "$COSMOS3_UV_ENV/bin/python" pynvml
fi

echo "Python: $COSMOS3_UV_ENV/bin/python"

## 4. Verify GPU and Python Environment

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "Missing $COSMOS3_UV_ENV/bin/python"
  echo "Run the Install cell first."
  exit 1
fi
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" "$COSMOS3_UV_ENV/bin/python" - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"device {index}:", torch.cuda.get_device_name(index))
PY

## 5. Download the PAI-Bench-C Dataset

PAI-Bench-C is hosted on Hugging Face at [`shi-labs/physical-ai-bench-conditional-generation`](https://huggingface.co/datasets/shi-labs/physical-ai-bench-conditional-generation). We download it by cloning with Git LFS.

Set `DOWNLOAD_MODE = "demo"` to download only the single demo task (fast, ~10 MB) for an end-to-end test. Set `"full"` to download all 600 tasks before running the full sweep.

Dataset layout (under `$PAIBENCH_C_DATASET_ROOT`):

```
physical-ai-bench-conditional-generation/
├── videos/           # 600 GT reference videos (task_0000.mp4 … task_0599.mp4)
├── canny/            # pre-computed Canny edge maps     (control input: edge)
├── blur/             # pre-computed blur references     (control input: blur)
├── depth_vids/       # pre-computed depth maps          (control input: depth)
├── sam2_vids/        # pre-computed SAM2 seg maps       (control input: seg)
├── depth_npzs/       # pre-cached GT depth arrays       (eval: skips re-running DepthAnything)
├── sam2_pkls/        # pre-cached GT seg masks          (eval: skips re-running SAM2)
├── captions/         # per-task captions (task_XXXX.json)
└── metadata.csv      # task index with paths and base captions
```

The control-signal videos (`canny/`, `blur/`, `depth_vids/`, `sam2_vids/`) are used as generation inputs. The `videos/` GT clips and pre-cached `depth_npzs/` / `sam2_pkls/` are used during metric evaluation.

In [ ]:
# "demo"  → download only the single DEMO_TASK_ID (fast, ~10 MB per task)
# "full"  → download all 600 tasks (required before running full sweeps)
DOWNLOAD_MODE = os.environ.get("PAIBENCH_C_DOWNLOAD_MODE", "demo")

os.environ["PAIBENCH_C_DOWNLOAD_MODE"] = DOWNLOAD_MODE
os.environ["DEMO_TASK_ID"] = DEMO_TASK_ID
print(f"DOWNLOAD_MODE = {DOWNLOAD_MODE}  (DEMO_TASK_ID = {DEMO_TASK_ID})")

In [ ]:
%%bash
set -euo pipefail

TASK="${DEMO_TASK_ID:-task_0000}"
NUM_TASKS="${DEMO_NUM_TASKS:-1}"
MODE="${PAIBENCH_C_DOWNLOAD_MODE:-demo}"

if ! command -v git-lfs >/dev/null 2>&1; then
  echo "git-lfs not found - downloading static binary ..."
  _lfs_bin_dir="${COSMOS3_CACHE_ROOT:-.cache}/bin"
  mkdir -p "$_lfs_bin_dir"
  _lfs_arch="$(uname -m)"
  [[ "$_lfs_arch" == "x86_64" ]] && _lfs_arch="amd64" || _lfs_arch="arm64"
  _lfs_ver="3.6.1"
  _lfs_url="https://github.com/git-lfs/git-lfs/releases/download/v${_lfs_ver}/git-lfs-linux-${_lfs_arch}-v${_lfs_ver}.tar.gz"
  curl -LsSf "$_lfs_url" \
      | tar -xz -C "$_lfs_bin_dir" --strip-components=1 "git-lfs-${_lfs_ver}/git-lfs" 2>/dev/null \
      || curl -LsSf "$_lfs_url" \
      | tar -xz -C "$_lfs_bin_dir" --wildcards --strip-components=1 "*/git-lfs"
  chmod +x "$_lfs_bin_dir/git-lfs"
  export PATH="$_lfs_bin_dir:$PATH"
  command -v git-lfs >/dev/null 2>&1 \
      || { echo "ERROR: git-lfs install failed. Install manually: https://git-lfs.com/"; exit 1; }
fi
echo "git-lfs: $(git-lfs version)"

# ── Clone (or reuse) ──────────────────────────────────────────────────────
if [ -d "$PAIBENCH_C_DATASET_ROOT/.git" ]; then
  echo "Repo already cloned at $PAIBENCH_C_DATASET_ROOT"
else
  echo "Cloning (without LFS blobs) …"
  git lfs install
  mkdir -p "$(dirname "$PAIBENCH_C_DATASET_ROOT")"
  GIT_LFS_SKIP_SMUDGE=1 \
  git -c credential.helper= clone "$PAIBENCH_C_HF_URL" "$PAIBENCH_C_DATASET_ROOT"
fi

cd "$PAIBENCH_C_DATASET_ROOT"

# ── Pull LFS objects ──────────────────────────────────────────────────────
if [ "$MODE" = "demo" ]; then
  # Parse the numeric suffix from TASK (e.g. task_0000 → 0) then pull NUM_TASKS tasks.
  BASE="${TASK%%_*}_"          # "task_"
  START="${TASK##*_}"          # "0000"
  START_N=$((10#$START))       # decimal, strips leading zeros
  echo "Demo mode: pulling ${NUM_TASKS} task(s) starting from ${TASK} …"
  for i in $(seq 0 $((NUM_TASKS - 1))); do
    T=$(printf "${BASE}%04d" $((START_N + i)))
    git lfs pull --include="videos/${T}.mp4"
    git lfs pull --include="canny/${T}.mp4"
    git lfs pull --include="blur/${T}.mp4"
    git lfs pull --include="depth_vids/${T}.mp4"
    git lfs pull --include="sam2_vids/${T}.mp4"
    git lfs pull --include="depth_npzs/${T}.npz"
    git lfs pull --include="sam2_pkls/${T}.pkl"
  done
  echo "Demo download complete (${NUM_TASKS} task(s))."
else
  echo "Full mode: pulling all LFS objects (this may take a while) …"
  git lfs pull
  echo "Full download complete."
fi

echo "--- $PAIBENCH_C_DATASET_ROOT ---"
ls "$PAIBENCH_C_DATASET_ROOT"

## 6. Load Prompts and Preview the Demo Case

We load the 600-task prompt file from `assets/prompts.json`. Each entry includes the JSON caption and the relative control-signal paths for all four modalities.

In [ ]:
PROMPTS_BY_TASK = {row["task_id"]: row for row in json.loads(PROMPTS_FILE.read_text())}

print(f"Loaded {len(PROMPTS_BY_TASK)} tasks from {PROMPTS_FILE.name}")
assert len(PROMPTS_BY_TASK) == 600, f"expected 600 tasks, got {len(PROMPTS_BY_TASK)}"

demo_entry = PROMPTS_BY_TASK[DEMO_TASK_ID]
print(f"\nDemo task: {DEMO_TASK_ID}")
print(f"Caption (first 200 chars): {demo_entry['caption'][:200]}...")
print(f"Control paths:")
for modality in ("canny_path", "blur_path", "depth_path", "seg_path"):
    print(f"  {modality}: {demo_entry[modality]}")

## 7. Helper Functions

**Recipe (shared by all modalities)**

- **Output length**: 121 frames at 30 fps.
- **Resolution**: 720p (auto aspect-ratio detected from the control video).
- **Sampling**: `num_steps=50`, `guidance=3.0`, `shift=10.0`, `seed=2026`. `control_guidance=1.5` for edge/blur/depth; `control_guidance=2.0` for seg.
- **Conditioning**: `num_conditional_frames=1`, `num_first_chunk_conditional_frames=0`, `negative_metadata_mode="none"`, `negative_prompt_keep_metadata=False`.
- **model_mode**: `"video2video"`.
- **Control key**: one of `edge`, `blur`, `depth`, `seg`, with the dataset's pre-computed video as `control_path`.
- **edge** uses `preset_edge_threshold="medium"`; **blur** uses `preset_blur_strength="medium"`.

Output naming: predictions are written as `task_XXXX/vision.mp4` and then collected into a flat `videos/task_XXXX.mp4` layout for metric evaluation.

In [ ]:
NUM_FRAMES = 121
FPS = 30
RESOLUTION = "720"
NUM_STEPS = 50
GUIDANCE = 3.0
SHIFT = 10.0
SEED = 2026

# control_guidance differs by modality (seg uses stronger guidance)
CONTROL_GUIDANCE = {
    "edge":  1.5,
    "blur":  1.5,
    "depth": 1.5,
    "seg":   2.0,
}

# aspect_ratio differs by modality (blur uses 4:3; others use 16:9)
ASPECT_RATIO = {
    "edge":  "16,9",
    "blur":  "4,3",
    "depth": "16,9",
    "seg":   "16,9",
}

MODALITY_CONTROL_KEY = {
    "edge":  "canny_path",
    "blur":  "blur_path",
    "depth": "depth_path",
    "seg":   "seg_path",
}


def control_video_path(task_id: str, modality: str) -> Path:
    """Resolve the control-signal video for a task and modality."""
    entry = PROMPTS_BY_TASK[task_id]
    rel_path = entry[MODALITY_CONTROL_KEY[modality]]
    full_path = PAIBENCH_C_DATASET_ROOT / rel_path
    if not full_path.exists():
        raise FileNotFoundError(f"Control video not found: {full_path}")
    return full_path


def build_v2v_row(task_id: str, modality: str) -> dict:
    """Build one JSONL row for video-to-video transfer inference."""
    entry = PROMPTS_BY_TASK[task_id]
    ctrl_path = control_video_path(task_id, modality)

    row: dict = {
        "name": task_id,
        "model_mode": "video2video",
        "resolution": RESOLUTION,
        "aspect_ratio": ASPECT_RATIO[modality],
        "num_frames": NUM_FRAMES,
        "fps": FPS,
        "shift": SHIFT,
        "num_steps": NUM_STEPS,
        "seed": SEED,
        "guidance": GUIDANCE,
        "control_guidance": CONTROL_GUIDANCE[modality],
        "num_outputs": 1,
        "share_vision_temporal_positions": True,
        "num_video_frames_per_chunk": NUM_FRAMES,
        "num_conditional_frames": 1,
        "num_first_chunk_conditional_frames": 0,
        "negative_metadata_mode": "none",
        "negative_prompt_keep_metadata": False,
        "prompt": entry["caption"],
        "negative_prompt": entry["negative_prompt"],
    }

    if modality == "edge":
        row["edge"] = {"control_path": str(ctrl_path), "preset_edge_threshold": "medium"}
    elif modality == "blur":
        row["blur"] = {"control_path": str(ctrl_path), "preset_blur_strength": "medium"}
    elif modality == "depth":
        row["depth"] = {"control_path": str(ctrl_path)}
    elif modality == "seg":
        row["seg"] = {"control_path": str(ctrl_path)}
    else:
        raise ValueError(f"Unknown modality: {modality}")

    return row


def build_input_jsonl(rows: list[dict], dst_jsonl: Path) -> Path:
    dst_jsonl.parent.mkdir(parents=True, exist_ok=True)
    with dst_jsonl.open("w") as fp:
        for row in rows:
            fp.write(json.dumps(row) + "\n")
    return dst_jsonl


def collect_outputs(raw_dir: Path, videos_dir: Path) -> int:
    """Flatten <raw_dir>/<name>/vision.mp4 -> <videos_dir>/<name>.mp4.

    The PAI-Bench-C evaluation expects a flat videos/ directory with
    task_XXXX.mp4 filenames.
    """
    videos_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for vision_mp4 in sorted(raw_dir.rglob("vision.mp4")):
        name = vision_mp4.parent.name
        dst = videos_dir / f"{name}.mp4"
        if not dst.exists():
            shutil.copy2(vision_mp4, dst)
        count += 1
    return count


def display_video(path: Path, width: int = 480) -> None:
    from IPython.display import Video, display
    display(Video(filename=str(path), embed=True, width=width))


print("Helper functions ready.")

## 8. Demo Case — Run One Task with One Modality

Runs a single transfer case (`DEMO_TASK_ID` with `DEMO_MODALITY`) to verify the pipeline end-to-end before launching the full 600-task sweep.

In [ ]:
# Build the list of demo task IDs: DEMO_NUM_TASKS consecutive tasks starting at DEMO_TASK_ID.
_start = int(DEMO_TASK_ID.split("_")[1])
DEMO_TASK_IDS = [f"task_{(_start + i):04d}" for i in range(DEMO_NUM_TASKS)]

demo_run_dir = PAIBENCH_C_OUTPUT_ROOT / "demo" / DEMO_MODALITY
demo_run_dir.mkdir(parents=True, exist_ok=True)

demo_input_jsonl = demo_run_dir / "input.jsonl"
demo_output_dir = demo_run_dir / "raw"
demo_output_dir.mkdir(parents=True, exist_ok=True)

build_input_jsonl([build_v2v_row(tid, DEMO_MODALITY) for tid in DEMO_TASK_IDS], demo_input_jsonl)

os.environ["DEMO_INPUT"] = str(demo_input_jsonl)
os.environ["DEMO_OUTPUT_DIR"] = str(demo_output_dir)

print(f"DEMO_TASK_IDS = {DEMO_TASK_IDS}")
print("DEMO_INPUT =", demo_input_jsonl)
print("DEMO_OUTPUT_DIR =", demo_output_dir)
print()
print("row preview:")
print(demo_input_jsonl.read_text()[:400], "...")

### Run Demo Inference

In [ ]:
%%bash
set -euo pipefail

# Unset vars that can contaminate the cosmos3 Python 3.13 workers when inherited
# from the Jupyter kernel (Python 3.12) environment.
unset PYTHONPATH PYTHONSTARTUP PYTHONHOME VIRTUAL_ENV CONDA_PREFIX CONDA_DEFAULT_ENV MPLBACKEND 2>/dev/null || true
# Redirect torch kernel/hub cache off the quota-limited home directory.
export TORCH_HOME="${TRITON_CACHE_DIR%/triton}"

cd "$COSMOS3_REPO"

# Capture per-rank output to a temp dir so the actual error is visible on failure.
_logdir="/tmp/cosmos_inference_$(date +%s)"
mkdir -p "$_logdir"

CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --standalone \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --log-dir "$_logdir" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$DEMO_INPUT" \
  -o "$DEMO_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails \
|| {
  echo ""
  echo "=== torchrun FAILED - dumping per-rank logs from $_logdir ==="
  ls "$_logdir/" 2>/dev/null || echo "(log dir is empty)"
  for f in "$_logdir"/*; do
    [ -f "$f" ] && { echo "--- $f ---"; head -80 "$f"; echo; }
  done
  exit 1
}

In [ ]:
raw_outputs = sorted(demo_output_dir.rglob("vision.mp4"))
if raw_outputs:
    raw_mp4 = raw_outputs[0]
    print("demo output:", raw_mp4)
    display_video(raw_mp4)
else:
    print("No vision.mp4 found — run the demo inference cell first.")

### 8b. Demo Evaluation

Runs `compute_metrics.py` on the single generated demo task to verify the full pipeline end-to-end.

The cell below automatically clones [`physical-ai-bench`](https://github.com/SHI-Labs/physical-ai-bench), installs its venv, and downloads evaluation checkpoints on first run. Subsequent runs skip these steps.

To use an existing checkout instead, set `PAIBENCH_EVAL_ROOT` before running the notebook:
```bash
export PAIBENCH_EVAL_ROOT=/path/to/physical-ai-bench/conditional_generation
```

In [ ]:
# ── Locate / auto-clone physical-ai-bench ────────────────────────────────────
# Default: clone next to this notebook (paibench_c/physical-ai-bench/).
# Override by setting PAIBENCH_EVAL_ROOT before running.
PAIBENCH_EVAL_ROOT = Path(
    os.environ.get(
        "PAIBENCH_EVAL_ROOT",
        PAIBENCH_C_ROOT / "physical-ai-bench" / "conditional_generation",
    )
).resolve()
_paibench_repo = PAIBENCH_EVAL_ROOT.parent  # …/physical-ai-bench
_paibench_git_url = os.environ.get(
    "PAIBENCH_EVAL_GIT_URL",
    "https://github.com/trungtpham/pai-bench-c-cosmos3.git",
)
_paibench_git_branch = os.environ.get("PAIBENCH_EVAL_GIT_BRANCH", "main")

import subprocess as _sp

if not PAIBENCH_EVAL_ROOT.exists():
    print(f"Cloning physical-ai-bench ({_paibench_git_branch}) → {_paibench_repo} ...")
    _paibench_repo.parent.mkdir(parents=True, exist_ok=True)
    _sp.run(
        ["git", "-c", "credential.helper=",
         "clone", "-b", _paibench_git_branch, _paibench_git_url, str(_paibench_repo)],
        check=True,
    )
    _sp.run(
        ["git", "-C", str(_paibench_repo), "-c", "credential.helper=",
         "submodule", "update", "--init", "--recursive"],
        check=True,
    )
    print("Clone complete.")

_eval_venv_torchrun = PAIBENCH_EVAL_ROOT / ".venv" / "bin" / "torchrun"
if not _eval_venv_torchrun.exists():
    print(f"Setting up physical-ai-bench venv at {PAIBENCH_EVAL_ROOT} ...")
    import subprocess as _sp
    # Unset UV_PROJECT_ENVIRONMENT so uv creates .venv inside PAIBENCH_EVAL_ROOT,
    # not in the cosmos3 venv (which is what the kernel env points to).
    _paibench_env = {k: v for k, v in os.environ.items() if k != "UV_PROJECT_ENVIRONMENT"}
    _sp.run(["uv", "sync"], cwd=str(PAIBENCH_EVAL_ROOT), check=True, env=_paibench_env)
    _sp.run(
        ["uv", "pip", "install", "-e", "third_party/Grounded-SAM-2"],
        cwd=str(PAIBENCH_EVAL_ROOT), check=True, env=_paibench_env,
    )
    # Detect current GPU arch so groundingdino only compiles for this device.
    try:
        _cuda_arch = _sp.run(
            [str(COSMOS3_UV_ENV / "bin" / "python"), "-c",
             "import torch; cc=torch.cuda.get_device_capability(0); print(f'{cc[0]}.{cc[1]}')"],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        if _cuda_arch:
            os.environ["TORCH_CUDA_ARCH_LIST"] = _cuda_arch
            print(f"  Set TORCH_CUDA_ARCH_LIST={_cuda_arch} for groundingdino build")
    except Exception:
        pass  # best-effort; groundingdino will auto-detect arches
    _sp.run(
        ["uv", "pip", "install", "--no-build-isolation", "-e",
         "third_party/Grounded-SAM-2/grounding_dino"],
        cwd=str(PAIBENCH_EVAL_ROOT), check=True, env=_paibench_env,
    )
    print("Venv setup complete.")

_eval_ckpt_marker = PAIBENCH_EVAL_ROOT / "checkpoint"
if not _eval_ckpt_marker.exists():
    print("Downloading evaluation checkpoints ...")
    _sp.run(["bash", "get_checkpoint.sh"], cwd=str(PAIBENCH_EVAL_ROOT), check=True)
    print("Checkpoints downloaded.")

# ── Flatten raw demo output and set env vars for the eval subprocess ──────────
demo_videos_dir = PAIBENCH_C_OUTPUT_ROOT / "demo" / DEMO_MODALITY / "videos"
n = collect_outputs(demo_output_dir, demo_videos_dir)
print(f"Collected {n} video(s) → {demo_videos_dir}")
if n == 0:
    raise RuntimeError(
        f"No generated videos found under {demo_output_dir}. "
        "Run the demo inference cell (Section 8) before evaluation."
    )

demo_metrics_out = PAIBENCH_C_OUTPUT_ROOT / "demo" / DEMO_MODALITY / "metrics.json"
DEMO_EVAL_NGPU = min(DEMO_NUM_TASKS, int(COSMOS3_NUM_GPUS))

os.environ["PAIBENCH_EVAL_ROOT"] = str(PAIBENCH_EVAL_ROOT)
os.environ["DEMO_GT"]            = str(PAIBENCH_C_DATASET_ROOT)
os.environ["DEMO_VIDEOS_DIR"]    = str(demo_videos_dir.parent)
os.environ["DEMO_METRICS_OUT"]   = str(demo_metrics_out)
os.environ["DEMO_EVAL_NGPU"]     = str(DEMO_EVAL_NGPU)

print(f"PAIBENCH_EVAL_ROOT = {PAIBENCH_EVAL_ROOT}")
print(f"GT dataset         = {PAIBENCH_C_DATASET_ROOT}")
print(f"Videos dir         = {demo_videos_dir}")
print(f"Metrics output     = {demo_metrics_out}")
print(f"DEMO_EVAL_NGPU     = {DEMO_EVAL_NGPU}  (tasks={DEMO_NUM_TASKS}, gpus={COSMOS3_NUM_GPUS})")

In [ ]:
%%bash
set -euo pipefail

cd "$PAIBENCH_EVAL_ROOT"

# Unset Jupyter's matplotlib backend — it is invalid outside the kernel process.
unset MPLBACKEND
# Redirect torch kernel cache off the quota-limited home directory.
export TORCH_HOME="${TRITON_CACHE_DIR%/triton}"

# Use DEMO_EVAL_NGPU (= min(DEMO_NUM_TASKS, total GPUs)) to avoid NCCL timeout
# when the number of tasks is smaller than the number of GPUs.
NGPU="${DEMO_EVAL_NGPU:-1}"
GPU_LIST=$(echo "$CUDA_VISIBLE_DEVICES" | tr ',' '\n' | head -n "$NGPU" | tr '\n' ',' | sed 's/,$//')
CUDA_VISIBLE_DEVICES="$GPU_LIST" \
.venv/bin/torchrun \
  --standalone --nproc_per_node "$NGPU" \
  compute_metrics.py calculate-metrics \
  --gt_path    "$DEMO_GT" \
  --videos_path "$DEMO_VIDEOS_DIR" \
  --output_path "$DEMO_METRICS_OUT" \
  --force_recompute_gt_seg \
  --force_recompute_gt_depth

echo ""
echo "=== Demo metrics ($DEMO_MODALITY / $DEMO_TASK_ID) ==="
cat "$DEMO_METRICS_OUT"

In [ ]:
if demo_metrics_out.exists():
    metrics = json.loads(demo_metrics_out.read_text())
    print(f"Demo evaluation results ({DEMO_MODALITY} / {DEMO_TASK_ID}):\n")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
else:
    print("metrics.json not found — run the evaluation cell above first.")

## 9. (Optional) Full Sweep — Edge (600 tasks)

Set `RUN_ALL_EDGE = True` to write a single JSONL covering all 600 edge tasks and run generation.

In [ ]:
RUN_ALL_EDGE = False  # set to True to enable the full 600-task edge sweep
# Requires DOWNLOAD_MODE="full" (or the full dataset already cloned).

if RUN_ALL_EDGE:
    task_ids = sorted(PROMPTS_BY_TASK.keys())
    assert len(task_ids) == 600

    edge_inputs_dir = PAIBENCH_C_OUTPUT_ROOT / "edge" / "inputs"
    edge_raw_dir = PAIBENCH_C_OUTPUT_ROOT / "edge" / "raw"
    edge_videos_dir = PAIBENCH_C_OUTPUT_ROOT / "edge" / "videos"

    edge_jsonl = edge_inputs_dir / "all_600.jsonl"
    build_input_jsonl([build_v2v_row(tid, "edge") for tid in task_ids], edge_jsonl)

    os.environ["EDGE_FULL_INPUT"] = str(edge_jsonl)
    os.environ["EDGE_FULL_OUTPUT_DIR"] = str(edge_raw_dir)
    os.environ["EDGE_VIDEOS_DIR"] = str(edge_videos_dir)
    print("EDGE_FULL_INPUT =", edge_jsonl)
    print("EDGE_FULL_OUTPUT_DIR =", edge_raw_dir)
    print(f"Wrote {len(task_ids)} rows. Run the next bash cell to generate all 600 edge videos.")

### Generate All Edge Outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${EDGE_FULL_INPUT:-}" ]; then
  echo "Set RUN_ALL_EDGE = True in the previous Python cell and re-run it first."
  exit 0
fi

unset PYTHONPATH PYTHONSTARTUP PYTHONHOME VIRTUAL_ENV CONDA_PREFIX CONDA_DEFAULT_ENV MPLBACKEND 2>/dev/null || true
export TORCH_HOME="${TRITON_CACHE_DIR%/triton}"

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --standalone \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$EDGE_FULL_INPUT" \
  -o "$EDGE_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

In [ ]:
if RUN_ALL_EDGE:
    edge_raw_dir = PAIBENCH_C_OUTPUT_ROOT / "edge" / "raw"
    edge_videos_dir = PAIBENCH_C_OUTPUT_ROOT / "edge" / "videos"
    n = collect_outputs(edge_raw_dir, edge_videos_dir)
    print(f"Collected {n} edge videos → {edge_videos_dir}")

## 10. (Optional) Full Sweep — Blur (600 tasks)

Set `RUN_ALL_BLUR = True` to run all 600 blur-conditioned tasks.

In [ ]:
RUN_ALL_BLUR = False  # set to True to enable the full 600-task blur sweep
# Requires DOWNLOAD_MODE="full" (or the full dataset already cloned).

if RUN_ALL_BLUR:
    task_ids = sorted(PROMPTS_BY_TASK.keys())

    blur_inputs_dir = PAIBENCH_C_OUTPUT_ROOT / "blur" / "inputs"
    blur_raw_dir = PAIBENCH_C_OUTPUT_ROOT / "blur" / "raw"
    blur_videos_dir = PAIBENCH_C_OUTPUT_ROOT / "blur" / "videos"

    blur_jsonl = blur_inputs_dir / "all_600.jsonl"
    build_input_jsonl([build_v2v_row(tid, "blur") for tid in task_ids], blur_jsonl)

    os.environ["BLUR_FULL_INPUT"] = str(blur_jsonl)
    os.environ["BLUR_FULL_OUTPUT_DIR"] = str(blur_raw_dir)
    os.environ["BLUR_VIDEOS_DIR"] = str(blur_videos_dir)
    print("BLUR_FULL_INPUT =", blur_jsonl)
    print(f"Wrote {len(task_ids)} rows. Run the next bash cell to generate all 600 blur videos.")

### Generate All Blur Outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${BLUR_FULL_INPUT:-}" ]; then
  echo "Set RUN_ALL_BLUR = True in the previous Python cell and re-run it first."
  exit 0
fi

unset PYTHONPATH PYTHONSTARTUP PYTHONHOME VIRTUAL_ENV CONDA_PREFIX CONDA_DEFAULT_ENV MPLBACKEND 2>/dev/null || true
export TORCH_HOME="${TRITON_CACHE_DIR%/triton}"

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --standalone \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$BLUR_FULL_INPUT" \
  -o "$BLUR_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

In [ ]:
if RUN_ALL_BLUR:
    blur_raw_dir = PAIBENCH_C_OUTPUT_ROOT / "blur" / "raw"
    blur_videos_dir = PAIBENCH_C_OUTPUT_ROOT / "blur" / "videos"
    n = collect_outputs(blur_raw_dir, blur_videos_dir)
    print(f"Collected {n} blur videos → {blur_videos_dir}")

## 11. (Optional) Full Sweep — Depth (600 tasks)

Set `RUN_ALL_DEPTH = True` to run all 600 depth-conditioned tasks.

In [ ]:
RUN_ALL_DEPTH = False  # set to True to enable the full 600-task depth sweep
# Requires DOWNLOAD_MODE="full" (or the full dataset already cloned).

if RUN_ALL_DEPTH:
    task_ids = sorted(PROMPTS_BY_TASK.keys())

    depth_inputs_dir = PAIBENCH_C_OUTPUT_ROOT / "depth" / "inputs"
    depth_raw_dir = PAIBENCH_C_OUTPUT_ROOT / "depth" / "raw"
    depth_videos_dir = PAIBENCH_C_OUTPUT_ROOT / "depth" / "videos"

    depth_jsonl = depth_inputs_dir / "all_600.jsonl"
    build_input_jsonl([build_v2v_row(tid, "depth") for tid in task_ids], depth_jsonl)

    os.environ["DEPTH_FULL_INPUT"] = str(depth_jsonl)
    os.environ["DEPTH_FULL_OUTPUT_DIR"] = str(depth_raw_dir)
    os.environ["DEPTH_VIDEOS_DIR"] = str(depth_videos_dir)
    print("DEPTH_FULL_INPUT =", depth_jsonl)
    print(f"Wrote {len(task_ids)} rows. Run the next bash cell to generate all 600 depth videos.")

### Generate All Depth Outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${DEPTH_FULL_INPUT:-}" ]; then
  echo "Set RUN_ALL_DEPTH = True in the previous Python cell and re-run it first."
  exit 0
fi

unset PYTHONPATH PYTHONSTARTUP PYTHONHOME VIRTUAL_ENV CONDA_PREFIX CONDA_DEFAULT_ENV MPLBACKEND 2>/dev/null || true
export TORCH_HOME="${TRITON_CACHE_DIR%/triton}"

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --standalone \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$DEPTH_FULL_INPUT" \
  -o "$DEPTH_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

In [ ]:
if RUN_ALL_DEPTH:
    depth_raw_dir = PAIBENCH_C_OUTPUT_ROOT / "depth" / "raw"
    depth_videos_dir = PAIBENCH_C_OUTPUT_ROOT / "depth" / "videos"
    n = collect_outputs(depth_raw_dir, depth_videos_dir)
    print(f"Collected {n} depth videos → {depth_videos_dir}")

## 12. (Optional) Full Sweep — Seg (600 tasks)

Set `RUN_ALL_SEG = True` to run all 600 segmentation-conditioned tasks.

In [ ]:
RUN_ALL_SEG = False  # set to True to enable the full 600-task seg sweep
# Requires DOWNLOAD_MODE="full" (or the full dataset already cloned).

if RUN_ALL_SEG:
    task_ids = sorted(PROMPTS_BY_TASK.keys())

    seg_inputs_dir = PAIBENCH_C_OUTPUT_ROOT / "seg" / "inputs"
    seg_raw_dir = PAIBENCH_C_OUTPUT_ROOT / "seg" / "raw"
    seg_videos_dir = PAIBENCH_C_OUTPUT_ROOT / "seg" / "videos"

    seg_jsonl = seg_inputs_dir / "all_600.jsonl"
    build_input_jsonl([build_v2v_row(tid, "seg") for tid in task_ids], seg_jsonl)

    os.environ["SEG_FULL_INPUT"] = str(seg_jsonl)
    os.environ["SEG_FULL_OUTPUT_DIR"] = str(seg_raw_dir)
    os.environ["SEG_VIDEOS_DIR"] = str(seg_videos_dir)
    print("SEG_FULL_INPUT =", seg_jsonl)
    print(f"Wrote {len(task_ids)} rows. Run the next bash cell to generate all 600 seg videos.")

### Generate All Seg Outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${SEG_FULL_INPUT:-}" ]; then
  echo "Set RUN_ALL_SEG = True in the previous Python cell and re-run it first."
  exit 0
fi

unset PYTHONPATH PYTHONSTARTUP PYTHONHOME VIRTUAL_ENV CONDA_PREFIX CONDA_DEFAULT_ENV MPLBACKEND 2>/dev/null || true
export TORCH_HOME="${TRITON_CACHE_DIR%/triton}"

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --standalone \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$SEG_FULL_INPUT" \
  -o "$SEG_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

In [ ]:
if RUN_ALL_SEG:
    seg_raw_dir = PAIBENCH_C_OUTPUT_ROOT / "seg" / "raw"
    seg_videos_dir = PAIBENCH_C_OUTPUT_ROOT / "seg" / "videos"
    n = collect_outputs(seg_raw_dir, seg_videos_dir)
    print(f"Collected {n} seg videos → {seg_videos_dir}")

## 13. Evaluate Generated Videos

After generating all prediction videos, run the [`physical-ai-bench`](https://github.com/SHI-Labs/physical-ai-bench) evaluation.

Each modality requires a `videos/` subdirectory containing `task_XXXX.mp4` files (one per task). The `collect_outputs` helper above flattens the raw inference output into this layout automatically.

> **Note:** Uses [`trungtpham/pai-bench-c-cosmos3`](https://github.com/trungtpham/pai-bench-c-cosmos3) (`main`), which contains fixes for depth SI-RMSE outlier blow-up and DOVER score bias (upstream [PR #8](https://github.com/SHI-Labs/physical-ai-bench/pull/8)). Auto-cloned on first run. To use an existing checkout, set `PAIBENCH_EVAL_ROOT` before running the notebook.

### Run evaluation (one modality at a time)

Set `NGPU` to the number of GPUs available (must match the node used for generation).

```bash
cd physical-ai-bench/conditional_generation
NGPU=4   # set to your GPU count
GT=/path/to/physical-ai-bench-conditional-generation
OUT=/path/to/paibench_c/outputs

unset MPLBACKEND   # prevent Jupyter's inline backend from leaking into subprocesses
for MOD in edge blur depth seg; do
  .venv/bin/torchrun --standalone --nproc_per_node $NGPU \
    compute_metrics.py calculate-metrics \
    --gt_path   $GT \
    --videos_path $OUT/$MOD \
    --output_path $OUT/${MOD}_metrics.json
done
```

`--videos_path` must contain a `videos/` subdirectory with `task_XXXX.mp4` files (one per task). The `collect_outputs` helper above creates this layout automatically.

### Reference scores (600 tasks each)

> Scores computed with [`trungtpham/pai-bench-c-cosmos3`](https://github.com/trungtpham/pai-bench-c-cosmos3) (`main`). The primary metric for each modality is **bold**; remaining columns are cross-modality diagnostics. SI-RMSE is lower-is-better; all others are higher-is-better.

#### Cosmos3-Nano

| Modality | Canny F1 | Blur SSIM | Depth SI-RMSE ↓ | Seg mIoU | Seg Recall | DOVER |
| -------- | -------: | --------: | --------------: | -------: | ---------: | ----: |
| edge     | **0.491** | 0.853 | 0.556 | 0.765 | 0.704 | 10.14 |
| blur     | 0.292 | **0.902** | 0.458 | 0.770 | 0.700 |  9.91 |
| depth    | 0.190 | 0.748 | **0.610** | 0.738 | 0.683 | 10.75 |
| seg      | 0.154 | 0.708 | 0.789 | **0.753** | 0.676 | 10.60 |

#### Cosmos3-Super

> Set `PAIBENCH_C_CHECKPOINT=Cosmos3-Super` to reproduce these scores.

| Modality | Canny F1 | Blur SSIM | Depth SI-RMSE ↓ | Seg mIoU | Seg Recall | DOVER |
| -------- | -------: | --------: | --------------: | -------: | ---------: | ----: |
| edge     | **0.502** | 0.863 | 0.529 | 0.764 | 0.713 | 10.09 |
| blur     | 0.306 | **0.906** | 0.437 | 0.770 | 0.705 |  9.98 |
| depth    | 0.200 | 0.758 | **0.560** | 0.756 | 0.687 | 10.62 |
| seg      | 0.159 | 0.709 | 0.747 | **0.751** | 0.683 | 10.62 |